# LogRMS 50M TinyStories systems benchmark

This self-contained Colab compares fused AdamW, foreach AdamW, torchao 8-bit AdamW, and compiled LogRMS on the same approximately 50M-parameter causal transformer. It records convergence, persistent optimizer state, optimizer-step time, full-step time, total wall time, throughput, and whole-process peak PyTorch allocation to CSV.


In [ ]:
import sys, subprocess

REQUIRED = [
    "datasets>=2.20,<4",
    "sentencepiece>=0.2,<1",
    "torchao==0.17.0",
    "psutil>=5.9",
    "pandas>=2.0",
    "matplotlib>=3.7",
    "tqdm>=4.66",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED])

In [ ]:
from __future__ import annotations

import base64
import csv
import gc
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import random
import time
import uuid
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterator

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm.auto import tqdm


@dataclass(reference=True)
class Config:
    dataset_id: str = "roneneldan/TinyStories"
    vocab_size: int = 4096
    sequence_length: int = 256
    d_model: int = 512
    heads: int = 8
    layers: int = 15
    feedforward_dim: int = 2048
    dropout: float = 0.0
    microbatch_size: int = 16
    gradient_accumulation_steps: int = 2
    eval_batch_size: int = 32
    epochs: int = 1
    seed: int = 2701
    data_seed: int = 3701
    peak_lr: float = 1e-3
    weight_decay: float = 0.01
    beta2: float = 0.999
    eps: float = 1e-8
    max_grad_norm: float = 1.0
    warmup_fraction: float = 0.05
    min_lr_ratio: float = 0.01
    eval_every: int = 5000
    eval_batches: int = 128
    checkpoint_every: int = 10000
    progress_update_every: int = 20
    timing_exclude_first_steps: int = 20
    num_workers: int = 2
    persist_data_cache_in_drive: bool = True
    force_rebuild_data: bool = False
    run_training: bool = True
    run_final_test: bool = True


CFG = Config()
NOTEBOOK_BUILD = "LOGRMS-50M-SPEED-BENCHMARK-20260713-A"
OPTIMIZERS = (
    "adamw_foreach",
    "adamw_fused",
    "adamw8bit_torchao",
    "logrms_compiled",
)
EXECUTION_PATHS = {
    "adamw_foreach": "torch.optim.AdamW foreach=True fused=False",
    "adamw_fused": "torch.optim.AdamW fused=True foreach=False",
    "adamw8bit_torchao": "torchao AdamW8bit internal torch.compile",
    "logrms_compiled": "LogRMS current Inductor parameter-group graph",
}

assert CFG.epochs == 1, "The registered speed protocol is one complete epoch."
assert CFG.d_model % CFG.heads == 0
print({"notebook_build": NOTEBOOK_BUILD, "optimizers": OPTIMIZERS})

## Storage, GPU, and reproducibility preflight

Results use a unique Drive directory. If the earlier 5M experiment's compatible
tokenizer and token cache are present, this notebook reuses them read-only.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
except Exception:
    MYDRIVE = Path("/content")

PROJECT_ROOT = MYDRIVE / "logrms_50m_speed_benchmark_current"
RUN_ROOT = PROJECT_ROOT / "runs"
LOCAL_ROOT = Path("/content/logrms_50m_speed_benchmark_current")
LEGACY_DATA_ROOT = MYDRIVE / "logrms_tinystories_5m" / "data"
NEW_DATA_ROOT = (PROJECT_ROOT if CFG.persist_data_cache_in_drive else LOCAL_ROOT) / "data"

required_legacy = [
    LEGACY_DATA_ROOT / "tinystories_bpe4096.model",
    LEGACY_DATA_ROOT / "train.uint16.bin",
    LEGACY_DATA_ROOT / "train.json",
    LEGACY_DATA_ROOT / "validation.uint16.bin",
    LEGACY_DATA_ROOT / "validation.json",
    LEGACY_DATA_ROOT / "test.uint16.bin",
    LEGACY_DATA_ROOT / "test.json",
]
DATA_ROOT = LEGACY_DATA_ROOT if all(path.exists() for path in required_legacy) else NEW_DATA_ROOT
for path in (PROJECT_ROOT, RUN_ROOT, LOCAL_ROOT, DATA_ROOT):
    path.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime: Runtime > Change runtime type > GPU")
if not hasattr(torch, "compile"):
    raise RuntimeError("This benchmark requires torch.compile")

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
SCALER_ENABLED = not BF16


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True), encoding="utf-8")
    os.replace(temporary, path)


ENVIRONMENT = {
    "notebook_build": NOTEBOOK_BUILD,
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "torchao": importlib.metadata.version("torchao"),
    "datasets": importlib.metadata.version("datasets"),
    "amp_dtype": str(AMP_DTYPE),
    "scaler_enabled": SCALER_ENABLED,
    "data_root": str(DATA_ROOT),
    "run_root": str(RUN_ROOT),
    "config": asdict(CFG),
    "execution_paths": EXECUTION_PATHS,
}
if ENVIRONMENT["torchao"] != "0.17.0":
    raise RuntimeError(f"Registered protocol requires torchao 0.17.0, found {ENVIRONMENT['torchao']}")
atomic_json(RUN_ROOT / "environment.json", ENVIRONMENT)
display(pd.Series({k: v for k, v in ENVIRONMENT.items() if k not in {"config", "execution_paths"}}))

## Complete official TinyStories token stream

A deterministic 4096-token SentencePiece model is trained once. The complete
training split is packed losslessly with EOS between stories. The official
validation split is deterministically divided into validation and held-out test
halves. Encoding uses progress bars and is skipped when the verified cache exists.

In [ ]:
TOKENIZER_PREFIX = DATA_ROOT / "tinystories_bpe4096"
TOKENIZER_MODEL = TOKENIZER_PREFIX.with_suffix(".model")
TOKENIZER_TEXT = DATA_ROOT / "tokenizer_training_text.txt"
TOKEN_LIMIT_CHARS = 50_000_000


def get_splits():
    training = load_dataset(CFG.dataset_id, split="train")
    official_validation = load_dataset(CFG.dataset_id, split="validation")
    midpoint = len(official_validation) // 2
    validation = official_validation.select(range(midpoint))
    test = official_validation.select(range(midpoint, len(official_validation)))
    return training, validation, test


TRAIN_SPLIT, VALIDATION_SPLIT, TEST_SPLIT = get_splits()


def build_tokenizer() -> spm.SentencePieceProcessor:
    if CFG.force_rebuild_data or not TOKENIZER_MODEL.exists():
        written = 0
        with TOKENIZER_TEXT.open("w", encoding="utf-8") as output:
            progress = tqdm(TRAIN_SPLIT, desc="Tokenizer text", unit="stories")
            for row in progress:
                remaining = TOKEN_LIMIT_CHARS - written
                if remaining <= 0:
                    break
                text = row["text"][:remaining].replace("\x00", " ")
                output.write(text + "\n")
                written += len(text)
        spm.SentencePieceTrainer.train(
            input=str(TOKENIZER_TEXT),
            model_prefix=str(TOKENIZER_PREFIX),
            vocab_size=CFG.vocab_size,
            model_type="bpe",
            character_coverage=1.0,
            pad_id=0,
            unk_id=1,
            eos_id=2,
            bos_id=-1,
            byte_fallback=True,
            num_threads=max(1, os.cpu_count() or 1),
            shuffle_input_sentence=False,
            input_sentence_size=0,
            minloglevel=2,
        )
    processor = spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))
    if processor.vocab_size() != CFG.vocab_size:
        raise RuntimeError("Tokenizer vocabulary does not match the registered protocol")
    return processor


SP = build_tokenizer()


def encode_split(dataset, name: str) -> dict[str, Any]:
    binary = DATA_ROOT / f"{name}.uint16.bin"
    metadata = DATA_ROOT / f"{name}.json"
    if binary.exists() and metadata.exists() and not CFG.force_rebuild_data:
        info = json.loads(metadata.read_text(encoding="utf-8"))
        if info.get("sha256") == sha256_file(binary):
            info["path"] = str(binary)
            return info

    temporary = binary.with_suffix(binary.suffix + ".tmp")
    token_count = 0
    chunks = range(0, len(dataset), 1024)
    with temporary.open("wb") as output:
        for start in tqdm(chunks, desc=f"Encode {name}", unit="chunks"):
            encoded = SP.encode(dataset[start : start + 1024]["text"], out_type=int)
            for ids in encoded:
                values = np.asarray([*ids, SP.eos_id()], dtype=np.uint16)
                values.tofile(output)
                token_count += values.size
    os.replace(temporary, binary)
    info = {
        "name": name,
        "stories": len(dataset),
        "tokens": int(token_count),
        "bytes": binary.stat().st_size,
        "sha256": sha256_file(binary),
        "path": str(binary),
    }
    atomic_json(metadata, info)
    return info


SPLIT_INFO = {
    "train": encode_split(TRAIN_SPLIT, "train"),
    "validation": encode_split(VALIDATION_SPLIT, "validation"),
    "test": encode_split(TEST_SPLIT, "test"),
}
display(pd.DataFrame(SPLIT_INFO).T[["stories", "tokens", "bytes", "sha256"]])

In [ ]:
class PackedTokenDataset(Dataset):
    def __init__(self, info: dict[str, Any], sequence_length: int):
        self.tokens = np.memmap(info["path"], mode="r", dtype=np.uint16)
        self.sequence_length = sequence_length
        self.target_count = max(0, len(self.tokens) - 1)
        self.block_count = math.ceil(self.target_count / sequence_length)

    def __len__(self) -> int:
        return self.block_count

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, int]:
        start = index * self.sequence_length
        stop = min(start + self.sequence_length + 1, len(self.tokens))
        raw = np.asarray(self.tokens[start:stop], dtype=np.int64)
        real_targets = max(0, raw.size - 1)
        values = np.full(self.sequence_length + 1, 0, dtype=np.int64)
        values[: raw.size] = raw
        inputs = torch.from_numpy(values[:-1].copy())
        labels = torch.from_numpy(values[1:].copy())
        labels[real_targets:] = -100
        return inputs, labels, real_targets


class FixedEpochSampler(Sampler[int]):
    def __init__(self, size: int, seed: int, epoch: int, start_index: int = 0):
        generator = torch.Generator().manual_seed(seed + epoch)
        self.order = torch.randperm(size, generator=generator).tolist()[start_index:]

    def __iter__(self) -> Iterator[int]:
        return iter(self.order)

    def __len__(self) -> int:
        return len(self.order)


TRAIN_DATA = PackedTokenDataset(SPLIT_INFO["train"], CFG.sequence_length)
VALID_DATA = PackedTokenDataset(SPLIT_INFO["validation"], CFG.sequence_length)
TEST_DATA = PackedTokenDataset(SPLIT_INFO["test"], CFG.sequence_length)

## 49.5M-parameter causal transformer

The model uses tied embeddings, pre-norm blocks, and PyTorch scaled-dot-product
attention. Parameters and gradients are FP32; forward/backward activations use
BF16 when available, otherwise FP16 with gradient scaling. The model itself is
intentionally not wrapped in `torch.compile`, isolating optimizer execution.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = CFG.heads
        self.head_dim = CFG.d_model // CFG.heads
        self.qkv = nn.Linear(CFG.d_model, 3 * CFG.d_model)
        self.proj = nn.Linear(CFG.d_model, CFG.d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, length, width = x.shape
        qkv = self.qkv(x).view(batch, length, 3, self.heads, self.head_dim)
        query, key, value = qkv.unbind(dim=2)
        query, key, value = (tensor.transpose(1, 2) for tensor in (query, key, value))
        attended = F.scaled_dot_product_attention(
            query,
            key,
            value,
            dropout_p=CFG.dropout if self.training else 0.0,
            is_causal=True,
        )
        return self.proj(attended.transpose(1, 2).contiguous().view(batch, length, width))


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(CFG.d_model)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(CFG.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(CFG.d_model, CFG.feedforward_dim),
            nn.GELU(),
            nn.Linear(CFG.feedforward_dim, CFG.d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class TinyStoriesLM50M(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(CFG.vocab_size, CFG.d_model)
        self.position_embedding = nn.Embedding(CFG.sequence_length, CFG.d_model)
        self.blocks = nn.ModuleList([Block() for _ in range(CFG.layers)])
        self.final_norm = nn.LayerNorm(CFG.d_model)
        self.output = nn.Linear(CFG.d_model, CFG.vocab_size, bias=False)
        self.apply(self._initialize)
        self.output.weight = self.token_embedding.weight

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(tokens.shape[1], device=tokens.device)
        hidden = self.token_embedding(tokens) + self.position_embedding(positions)
        for block in self.blocks:
            hidden = block(hidden)
        return self.output(self.final_norm(hidden))


def state_sha256(state: dict[str, torch.Tensor]) -> str:
    digest = hashlib.sha256()
    for name, tensor in state.items():
        digest.update(name.encode("utf-8"))
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


seed_everything(CFG.seed)
template = TinyStoriesLM50M()
PARAMETER_COUNT = sum(parameter.numel() for parameter in template.parameters())
if PARAMETER_COUNT != 49_515_008:
    raise RuntimeError(f"Expected exactly 49,515,008 parameters, found {PARAMETER_COUNT:,}")
INITIAL_STATE = {name: tensor.detach().cpu().clone() for name, tensor in template.state_dict().items()}
INITIAL_STATE_SHA256 = state_sha256(INITIAL_STATE)
del template
gc.collect()
print({"parameter_count": PARAMETER_COUNT, "initial_state_sha256": INITIAL_STATE_SHA256})

In [ ]:
# Current LogRMS reference source.
REFERENCE_SOURCE_B64 = '...'

# Current LogRMS compiled source.
COMPILED_SOURCE_B64 = '...'


In [ ]:
SOURCE_ROOT = LOCAL_ROOT / "embedded_optimizer"
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_PATH = SOURCE_ROOT / "logrms_reference.py"
COMPILED_PATH = SOURCE_ROOT / "logrms_compiled.py"
REFERENCE_SHA256 = "1a59b6b948d565c646a435ad9c97f84b95579c347b981a32ebd81b119858e4f4"
COMPILED_SHA256 = "7c458787258bd9db1a4350e8e9cff8cc506ebfe78b859acdb8cbcbdbd3c96b98"
REFERENCE_PATH.write_bytes(base64.b64decode(REFERENCE_SOURCE_B64))
COMPILED_PATH.write_bytes(base64.b64decode(COMPILED_SOURCE_B64))
if sha256_file(REFERENCE_PATH) != REFERENCE_SHA256:
    raise RuntimeError("Embedded LogRMS reference source hash mismatch")
if sha256_file(COMPILED_PATH) != COMPILED_SHA256:
    raise RuntimeError("Embedded LogRMS compiled source hash mismatch")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from logrms_compiled import LogRMSCompiled
from logrms_reference import CONFIDENCE_DEFAULT
from torchao.optim import AdamW8bit
from torchao.optim.subclass_8bit import OptimState8bit


## Locked optimizer construction and state accounting

Global norm clipping is performed externally exactly once for every optimizer.
LogRMS therefore uses `max_norm=None`; this changes only where clipping occurs,
not the update mathematics. Compile failure fallback is disabled so a run cannot
silently become eager.

In [ ]:
def parameter_groups(model: nn.Module) -> list[dict[str, Any]]:
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    return [
        {"params": decay, "weight_decay": CFG.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]


def make_optimizer(name: str, model: nn.Module):
    groups = parameter_groups(model)
    common = dict(betas=(0.9, CFG.beta2), eps=CFG.eps)
    if name == "adamw_foreach":
        return torch.optim.AdamW(
            groups, lr=CFG.peak_lr, foreach=True, fused=False, **common
        )
    if name == "adamw_fused":
        return torch.optim.AdamW(
            groups, lr=CFG.peak_lr, foreach=False, fused=True, **common
        )
    if name == "adamw8bit_torchao":
        tensor_lr = torch.tensor(CFG.peak_lr, dtype=torch.float32, device=DEVICE)
        optimizer = AdamW8bit(
            groups,
            lr=tensor_lr,
            betas=(0.9, CFG.beta2),
            eps=CFG.eps,
            block_size=256,
        )
        if not all(torch.is_tensor(group["lr"]) for group in optimizer.param_groups):
            raise RuntimeError("torchao AdamW8bit did not preserve tensor learning rates")
        return optimizer
    if name == "logrms_compiled":
        return LogRMSCompiled(
            groups,
            lr=CFG.peak_lr,
            beta2=CFG.beta2,
            eps=CFG.eps,
            backend="8bit",
            confidence_table=CONFIDENCE_DEFAULT,
            max_norm=None,
            # External clip_grad_norm_ above performs the shared finite check;
            # disable the reference optimizer's otherwise redundant inf-norm pass.
            error_if_nonfinite=False,
            stochastic_rounding=True,
            rounding_seed=CFG.seed,
            compile_backend="inductor",
            compile_mode="reduce-overhead",
            fullgraph=True,
            dynamic=False,
            fallback_on_compile_error=False,
        )
    raise KeyError(name)


def set_optimizer_lr(optimizer, learning_rate: float) -> None:
    for group in optimizer.param_groups:
        if torch.is_tensor(group["lr"]):
            group["lr"].fill_(learning_rate)
        else:
            group["lr"] = float(learning_rate)


def scheduled_lr(step: int, total_steps: int) -> float:
    warmup_steps = max(1, round(total_steps * CFG.warmup_fraction))
    if step < warmup_steps:
        return CFG.peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps - 1)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return CFG.peak_lr * (CFG.min_lr_ratio + (1.0 - CFG.min_lr_ratio) * cosine)


def optimizer_state_audit(optimizer) -> tuple[int, float]:
    """Count unique optimizer-state tensor storage, including torchao subclasses."""
    seen: set[tuple[str, int, int]] = set()
    total_bytes = 0
    quantized_elements = 0

    def add(tensor: torch.Tensor) -> None:
        nonlocal total_bytes
        storage = tensor.untyped_storage()
        key = (str(tensor.device), storage.data_ptr(), storage.nbytes())
        if key not in seen:
            seen.add(key)
            total_bytes += storage.nbytes()

    for parameter, state in optimizer.state.items():
        if not isinstance(state, dict):
            continue
        if isinstance(state.get("exp_avg"), OptimState8bit):
            quantized_elements += parameter.numel()
        for value in state.values():
            if isinstance(value, OptimState8bit):
                add(value.codes)
                add(value.scale)
                add(value.qmap)
            elif torch.is_tensor(value):
                add(value)
    return total_bytes, quantized_elements / PARAMETER_COUNT


def dynamo_snapshot() -> dict[str, int]:
    snapshot: dict[str, int] = {}
    for category, counter in torch._dynamo.utils.counters.items():
        for key, value in counter.items():
            if isinstance(value, (int, np.integer)):
                snapshot[f"{category}.{key}"] = int(value)
    return snapshot


def counter_delta(before: dict[str, int], after: dict[str, int], key: str) -> int:
    return after.get(key, 0) - before.get(key, 0)

## Evaluation and durable CSV logging

In [ ]:
@torch.inference_mode()
def evaluate(model: nn.Module, dataset: Dataset, max_batches: int | None) -> dict[str, float]:
    model.eval()
    loader = DataLoader(
        dataset,
        batch_size=CFG.eval_batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )
    total_loss, total_tokens = 0.0, 0
    for batch_index, (inputs, labels, real_tokens) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(inputs)
            loss_sum = F.cross_entropy(
                logits.flatten(0, 1),
                labels.flatten(),
                ignore_index=-100,
                reduction="sum",
            )
        total_loss += float(loss_sum)
        total_tokens += int(real_tokens.sum())
    loss = total_loss / total_tokens
    return {
        "loss": loss,
        "perplexity": math.exp(min(20.0, loss)),
        "bits_per_token": loss / math.log(2),
        "tokens": total_tokens,
    }


class CSVWriter:
    def __init__(self, path: Path, flush_every: int = 100):
        self.path = path
        self.flush_every = flush_every
        self.handle = None
        self.writer = None
        self.rows_since_flush = 0

    def write(self, row: dict[str, Any]) -> None:
        if self.handle is None:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            exists = self.path.exists() and self.path.stat().st_size > 0
            self.handle = self.path.open("a", newline="", encoding="utf-8")
            self.writer = csv.DictWriter(self.handle, fieldnames=list(row))
            if not exists:
                self.writer.writeheader()
        self.writer.writerow(row)
        self.rows_since_flush += 1
        if self.rows_since_flush >= self.flush_every:
            self.handle.flush()
            self.rows_since_flush = 0

    def close(self) -> None:
        if self.handle is not None:
            self.handle.flush()
            self.handle.close()


def trim_csv_to_step(path: Path, global_step: int) -> None:
    if not path.exists() or path.stat().st_size == 0:
        return
    frame = pd.read_csv(path)
    if "global_step" in frame:
        frame = frame[frame.global_step <= global_step]
    frame.to_csv(path, index=False)


def atomic_torch_save(value: Any, path: Path) -> int:
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    os.replace(temporary, path)
    return path.stat().st_size

## One-epoch timed training run

Timing definitions:

- `full_step_wall_ms`: both accumulated forward/backward microbatches, clipping,
  optimizer update, scaler update, and required CUDA synchronization.
- `optimizer_step_wall_ms`: only `scaler.step(optimizer)` through completion.
- `optimizer_step_gpu_ms`: CUDA-event time around that optimizer call.
- `total_wall_seconds`: active runtime from model construction through final
  validation/test, including cold compilation, periodic evaluation, CSV work,
  and checkpoint I/O. Colab disconnection downtime is not counted.

In [ ]:
def train_run(name: str) -> dict[str, Any]:
    out = RUN_ROOT / name
    out.mkdir(parents=True, exist_ok=True)
    summary_path = out / "summary.json"
    checkpoint_path = out / "checkpoint.pt"
    progress_path = out / "progress.json"
    steps_path = out / "steps.csv"
    eval_path = out / "evaluations.csv"
    checkpoints_path = out / "checkpoints.csv"
    if summary_path.exists():
        return json.loads(summary_path.read_text(encoding="utf-8"))

    session_id = uuid.uuid4().hex[:12]
    session_start = time.perf_counter()
    seed_everything(CFG.seed)
    model = TinyStoriesLM50M().to(DEVICE)
    model.load_state_dict(INITIAL_STATE)
    optimizer = make_optimizer(name, model)
    scaler = torch.amp.GradScaler("cuda", enabled=SCALER_ENABLED)

    microbatches_per_epoch = math.ceil(len(TRAIN_DATA) / CFG.microbatch_size)
    updates_per_epoch = math.ceil(microbatches_per_epoch / CFG.gradient_accumulation_steps)
    total_updates = CFG.epochs * updates_per_epoch
    epoch = 0
    next_microbatch = 0
    global_step = 0
    tokens_processed = 0
    best_validation_loss = float("inf")
    best_validation_step = -1
    accumulated_active_wall = 0.0
    periodic_eval_seconds = 0.0
    checkpoint_seconds = 0.0
    checkpoint_bytes = 0

    if checkpoint_path.exists():
        state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        epoch = int(state["epoch"])
        next_microbatch = int(state["next_microbatch"])
        global_step = int(state["global_step"])
        tokens_processed = int(state["tokens_processed"])
        best_validation_loss = float(state["best_validation_loss"])
        best_validation_step = int(state["best_validation_step"])
        torch.set_rng_state(state["torch_rng"].cpu())
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda_rng"]])
        accumulated_active_wall = float(state.get("accumulated_active_wall", 0.0))
        periodic_eval_seconds = float(state.get("periodic_eval_seconds", 0.0))
        checkpoint_seconds = float(state.get("checkpoint_seconds", 0.0))
        if progress_path.exists():
            progress = json.loads(progress_path.read_text(encoding="utf-8"))
            if int(progress.get("global_step", -1)) == global_step:
                accumulated_active_wall = float(progress["accumulated_active_wall"])
                checkpoint_seconds = float(progress["checkpoint_seconds"])
                checkpoint_bytes = int(progress.get("checkpoint_bytes", 0))
        trim_csv_to_step(steps_path, global_step)
        trim_csv_to_step(eval_path, global_step)
        trim_csv_to_step(checkpoints_path, global_step)

    step_writer = CSVWriter(steps_path, flush_every=100)
    eval_writer = CSVWriter(eval_path, flush_every=1)
    checkpoint_writer = CSVWriter(checkpoints_path, flush_every=1)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    dynamo_before = dynamo_snapshot()
    # Use the function-entry timestamp so total wall time includes model and
    # optimizer construction, checkpoint restoration, and cold compiler setup.
    session_active_origin = session_start

    progress_bar = tqdm(
        total=total_updates,
        initial=global_step,
        desc=name,
        unit="updates",
        dynamic_ncols=True,
    )

    while epoch < CFG.epochs and global_step < total_updates:
        start_example = next_microbatch * CFG.microbatch_size
        sampler = FixedEpochSampler(len(TRAIN_DATA), CFG.data_seed, epoch, start_example)
        loader = DataLoader(
            TRAIN_DATA,
            batch_size=CFG.microbatch_size,
            sampler=sampler,
            num_workers=CFG.num_workers,
            pin_memory=True,
            drop_last=False,
            persistent_workers=CFG.num_workers > 0,
        )
        remaining_microbatches = microbatches_per_epoch - next_microbatch
        original_batch_index = next_microbatch
        loader_iterator = iter(loader)

        while remaining_microbatches > 0 and global_step < total_updates:
            accumulation_count = min(CFG.gradient_accumulation_steps, remaining_microbatches)
            learning_rate = scheduled_lr(global_step, total_updates)
            set_optimizer_lr(optimizer, learning_rate)
            optimizer.zero_grad(set_to_none=True)
            step_wall_start = time.perf_counter()
            full_start_event = torch.cuda.Event(enable_timing=True)
            backward_end_event = torch.cuda.Event(enable_timing=True)
            clip_end_event = torch.cuda.Event(enable_timing=True)
            optimizer_start_event = torch.cuda.Event(enable_timing=True)
            optimizer_end_event = torch.cuda.Event(enable_timing=True)
            full_end_event = torch.cuda.Event(enable_timing=True)
            full_start_event.record()

            loss_records: list[tuple[torch.Tensor, int]] = []
            update_tokens = 0
            for _ in range(accumulation_count):
                inputs, labels, real_tokens = next(loader_iterator)
                inputs = inputs.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    logits = model(inputs)
                    raw_loss = F.cross_entropy(
                        logits.flatten(0, 1), labels.flatten(), ignore_index=-100
                    )
                    scaled_loss = raw_loss / accumulation_count
                scaler.scale(scaled_loss).backward()
                token_count = int(real_tokens.sum())
                # Defer device-to-host loss conversion until the update's
                # deliberate timing synchronization. Converting here would
                # insert an extra CUDA synchronization for every microbatch.
                loss_records.append((raw_loss.detach(), token_count))
                update_tokens += token_count
                original_batch_index += 1
                remaining_microbatches -= 1

            backward_end_event.record()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), CFG.max_grad_norm, error_if_nonfinite=True
            )
            clip_end_event.record()
            old_scale = scaler.get_scale()
            optimizer_wall_start = time.perf_counter()
            optimizer_start_event.record()
            scaler.step(optimizer)
            optimizer_end_event.record()
            optimizer_end_event.synchronize()
            optimizer_step_wall_ms = (time.perf_counter() - optimizer_wall_start) * 1000.0
            scaler.update()
            full_end_event.record()
            full_end_event.synchronize()
            full_step_wall_ms = (time.perf_counter() - step_wall_start) * 1000.0
            skipped = scaler.get_scale() < old_scale

            global_step += 1
            next_microbatch = original_batch_index
            tokens_processed += update_tokens
            mean_loss = sum(float(loss_value) * count for loss_value, count in loss_records) / max(1, update_tokens)
            active_wall = accumulated_active_wall + (time.perf_counter() - session_active_origin)
            optimizer_gpu_ms = optimizer_start_event.elapsed_time(optimizer_end_event)
            backward_gpu_ms = full_start_event.elapsed_time(backward_end_event)
            clip_gpu_ms = backward_end_event.elapsed_time(clip_end_event)
            full_gpu_ms = full_start_event.elapsed_time(full_end_event)

            step_writer.write({
                "notebook_build": NOTEBOOK_BUILD,
                "session_id": session_id,
                "optimizer": name,
                "execution_path": EXECUTION_PATHS[name],
                "epoch": epoch,
                "global_step": global_step,
                "microbatch_end": next_microbatch,
                "accumulation_count": accumulation_count,
                "tokens_this_step": update_tokens,
                "tokens_processed": tokens_processed,
                "learning_rate": learning_rate,
                "train_loss": mean_loss,
                "grad_norm_preclip": float(grad_norm),
                "amp_skipped": skipped,
                "full_step_wall_ms": full_step_wall_ms,
                "full_step_gpu_ms": full_gpu_ms,
                "forward_backward_gpu_ms": backward_gpu_ms,
                "gradient_clip_gpu_ms": clip_gpu_ms,
                "optimizer_step_wall_ms": optimizer_step_wall_ms,
                "optimizer_step_gpu_ms": optimizer_gpu_ms,
                "tokens_per_second_step_wall": update_tokens / max(full_step_wall_ms / 1000.0, 1e-12),
                "run_active_wall_seconds": active_wall,
                "gpu_allocated_bytes": torch.cuda.memory_allocated(),
                "gpu_reserved_bytes": torch.cuda.memory_reserved(),
                "gpu_peak_allocated_bytes": torch.cuda.max_memory_allocated(),
                "cpu_rss_bytes": psutil.Process().memory_info().rss,
            })
            progress_bar.update(1)
            if global_step == 1 or global_step % CFG.progress_update_every == 0:
                progress_bar.set_postfix(
                    loss=f"{mean_loss:.3f}",
                    step_ms=f"{full_step_wall_ms:.1f}",
                    opt_ms=f"{optimizer_step_wall_ms:.1f}",
                    tok_s=f"{update_tokens / max(full_step_wall_ms / 1000.0, 1e-12):,.0f}",
                    refresh=False,
                )

            if global_step % CFG.eval_every == 0 or global_step == total_updates:
                evaluation_start = time.perf_counter()
                metrics = evaluate(model, VALID_DATA, CFG.eval_batches)
                evaluation_seconds = time.perf_counter() - evaluation_start
                periodic_eval_seconds += evaluation_seconds
                if metrics["loss"] < best_validation_loss:
                    best_validation_loss = metrics["loss"]
                    best_validation_step = global_step
                eval_writer.write({
                    "session_id": session_id,
                    "optimizer": name,
                    "global_step": global_step,
                    "tokens_processed": tokens_processed,
                    "validation_loss": metrics["loss"],
                    "validation_perplexity": metrics["perplexity"],
                    "validation_bits_per_token": metrics["bits_per_token"],
                    "validation_tokens": metrics["tokens"],
                    "evaluation_wall_seconds": evaluation_seconds,
                    "best_validation_loss": best_validation_loss,
                    "best_validation_step": best_validation_step,
                })
                progress_bar.set_postfix(
                    loss=f"{mean_loss:.3f}",
                    val=f"{metrics['loss']:.3f}",
                    step_ms=f"{full_step_wall_ms:.1f}",
                    opt_ms=f"{optimizer_step_wall_ms:.1f}",
                    refresh=False,
                )

            if global_step % CFG.checkpoint_every == 0 and global_step < total_updates:
                checkpoint_start = time.perf_counter()
                checkpoint_bytes = atomic_torch_save(
                    {
                        "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scaler": scaler.state_dict(),
                        "epoch": epoch,
                        "next_microbatch": next_microbatch,
                        "global_step": global_step,
                        "tokens_processed": tokens_processed,
                        "best_validation_loss": best_validation_loss,
                        "best_validation_step": best_validation_step,
                        "torch_rng": torch.get_rng_state(),
                        "cuda_rng": torch.cuda.get_rng_state_all(),
                        "accumulated_active_wall": accumulated_active_wall + (time.perf_counter() - session_active_origin),
                        "periodic_eval_seconds": periodic_eval_seconds,
                        "checkpoint_seconds": checkpoint_seconds,
                    },
                    checkpoint_path,
                )
                checkpoint_wall = time.perf_counter() - checkpoint_start
                checkpoint_seconds += checkpoint_wall
                checkpoint_writer.write({
                    "session_id": session_id,
                    "optimizer": name,
                    "global_step": global_step,
                    "checkpoint_wall_seconds": checkpoint_wall,
                    "checkpoint_bytes": checkpoint_bytes,
                })
                atomic_json(
                    progress_path,
                    {
                        "global_step": global_step,
                        "accumulated_active_wall": accumulated_active_wall + (time.perf_counter() - session_active_origin),
                        "checkpoint_seconds": checkpoint_seconds,
                        "checkpoint_bytes": checkpoint_bytes,
                    },
                )

        epoch += 1
        next_microbatch = 0

    progress_bar.close()
    step_writer.close()
    eval_writer.close()
    checkpoint_writer.close()

    final_validation_start = time.perf_counter()
    final_validation = evaluate(model, VALID_DATA, None)
    final_validation_seconds = time.perf_counter() - final_validation_start
    if CFG.run_final_test:
        final_test_start = time.perf_counter()
        final_test = evaluate(model, TEST_DATA, None)
        final_test_seconds = time.perf_counter() - final_test_start
    else:
        final_test = {"loss": None, "perplexity": None, "bits_per_token": None, "tokens": 0}
        final_test_seconds = 0.0

    total_wall_seconds = accumulated_active_wall + (time.perf_counter() - session_active_origin)
    state_bytes, quantized_fraction = optimizer_state_audit(optimizer)
    dynamo_after = dynamo_snapshot()
    step_frame = pd.read_csv(steps_path)
    steady = step_frame[step_frame.global_step > CFG.timing_exclude_first_steps]
    if steady.empty:
        steady = step_frame

    result = {
        "notebook_build": NOTEBOOK_BUILD,
        "optimizer": name,
        "execution_path": EXECUTION_PATHS[name],
        "gpu": GPU_NAME,
        "torch": torch.__version__,
        "torchao": ENVIRONMENT["torchao"],
        "parameter_count": PARAMETER_COUNT,
        "initial_state_sha256": INITIAL_STATE_SHA256,
        "epochs": CFG.epochs,
        "global_steps": global_step,
        "tokens_processed": tokens_processed,
        "peak_lr": CFG.peak_lr,
        "effective_batch_tokens": CFG.microbatch_size * CFG.gradient_accumulation_steps * CFG.sequence_length,
        "total_wall_seconds": total_wall_seconds,
        "periodic_eval_wall_seconds": periodic_eval_seconds,
        "final_validation_wall_seconds": final_validation_seconds,
        "final_test_wall_seconds": final_test_seconds,
        "checkpoint_wall_seconds": checkpoint_seconds,
        "measured_step_wall_seconds_sum": float(step_frame.full_step_wall_ms.sum() / 1000.0),
        "other_wall_seconds": total_wall_seconds
        - periodic_eval_seconds
        - final_validation_seconds
        - final_test_seconds
        - checkpoint_seconds
        - float(step_frame.full_step_wall_ms.sum() / 1000.0),
        "first_full_step_wall_ms": float(step_frame.iloc[0].full_step_wall_ms),
        "first_optimizer_step_wall_ms": float(step_frame.iloc[0].optimizer_step_wall_ms),
        "median_full_step_wall_ms": float(steady.full_step_wall_ms.median()),
        "mean_full_step_wall_ms": float(steady.full_step_wall_ms.mean()),
        "p95_full_step_wall_ms": float(steady.full_step_wall_ms.quantile(0.95)),
        "median_optimizer_step_wall_ms": float(steady.optimizer_step_wall_ms.median()),
        "mean_optimizer_step_wall_ms": float(steady.optimizer_step_wall_ms.mean()),
        "p95_optimizer_step_wall_ms": float(steady.optimizer_step_wall_ms.quantile(0.95)),
        "median_optimizer_step_gpu_ms": float(steady.optimizer_step_gpu_ms.median()),
        "median_step_tokens_per_second": float(steady.tokens_per_second_step_wall.median()),
        "end_to_end_tokens_per_second": tokens_processed / max(total_wall_seconds, 1e-12),
        "best_periodic_validation_loss": best_validation_loss,
        "best_periodic_validation_step": best_validation_step,
        "final_validation_loss": final_validation["loss"],
        "final_validation_perplexity": final_validation["perplexity"],
        "test_loss": final_test["loss"],
        "test_perplexity": final_test["perplexity"],
        "state_tensor_bytes": state_bytes,
        "state_bytes_per_parameter": state_bytes / PARAMETER_COUNT,
        "quantized_parameter_fraction": quantized_fraction,
        "peak_gpu_allocated_bytes": torch.cuda.max_memory_allocated(),
        "peak_gpu_reserved_bytes": torch.cuda.max_memory_reserved(),
        "last_checkpoint_bytes": checkpoint_bytes,
        "dynamo_unique_graphs_delta": counter_delta(dynamo_before, dynamo_after, "stats.unique_graphs"),
        "dynamo_graph_breaks_delta": counter_delta(dynamo_before, dynamo_after, "graph_break.total"),
        "compile_failures": int(getattr(optimizer, "compile_failures", 0)),
        "compilation_active": bool(getattr(optimizer, "compilation_active", False)) if name == "logrms_compiled" else None,
        "current_session_wall_seconds": time.perf_counter() - session_start,
    }
    if name == "logrms_compiled" and (result["compile_failures"] != 0 or not result["compilation_active"]):
        raise RuntimeError("LogRMS compilation was not active for the completed run")
    atomic_json(summary_path, result)
    if checkpoint_path.exists():
        checkpoint_path.unlink()
    if progress_path.exists():
        progress_path.unlink()
    del model, optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return result

## Run all four implementations

Completed optimizers are loaded from their summaries. Interrupted optimizers
resume from their latest durable update boundary.

In [ ]:
if CFG.run_training:
    RESULTS = [train_run(name) for name in OPTIMIZERS]
else:
    RESULTS = [json.loads((RUN_ROOT / name / "summary.json").read_text()) for name in OPTIMIZERS]

summary = pd.DataFrame(RESULTS)
summary.to_csv(RUN_ROOT / "run_summary.csv", index=False)
pd.concat(
    [pd.read_csv(RUN_ROOT / name / "steps.csv") for name in OPTIMIZERS],
    ignore_index=True,
).to_csv(RUN_ROOT / "all_steps.csv", index=False)
pd.concat(
    [pd.read_csv(RUN_ROOT / name / "evaluations.csv") for name in OPTIMIZERS],
    ignore_index=True,
).to_csv(RUN_ROOT / "all_evaluations.csv", index=False)
checkpoint_frames = [
    pd.read_csv(RUN_ROOT / name / "checkpoints.csv")
    for name in OPTIMIZERS
    if (RUN_ROOT / name / "checkpoints.csv").exists()
]
if checkpoint_frames:
    pd.concat(checkpoint_frames, ignore_index=True).to_csv(
        RUN_ROOT / "all_checkpoints.csv", index=False
    )
display(
    summary[
        [
            "optimizer",
            "total_wall_seconds",
            "median_full_step_wall_ms",
            "median_optimizer_step_wall_ms",
            "p95_optimizer_step_wall_ms",
            "median_step_tokens_per_second",
            "end_to_end_tokens_per_second",
            "state_bytes_per_parameter",
            "peak_gpu_allocated_bytes",
            "final_validation_loss",
            "test_loss",
        ]
    ].sort_values("total_wall_seconds")
)

## Timing and convergence plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ordered = summary.sort_values("median_optimizer_step_wall_ms")
axes[0].bar(ordered.optimizer, ordered.median_optimizer_step_wall_ms)
axes[0].set(title="Median optimizer step", ylabel="wall milliseconds")
axes[0].tick_params(axis="x", rotation=25)

ordered_total = summary.sort_values("total_wall_seconds")
axes[1].bar(ordered_total.optimizer, ordered_total.total_wall_seconds / 60.0)
axes[1].set(title="Total measured run wall time", ylabel="minutes")
axes[1].tick_params(axis="x", rotation=25)

for optimizer_name in OPTIMIZERS:
    evaluations = pd.read_csv(RUN_ROOT / optimizer_name / "evaluations.csv")
    axes[2].plot(
        evaluations.tokens_processed,
        evaluations.validation_loss,
        marker="o",
        label=optimizer_name,
    )
axes[2].set(title="Validation trajectory", xlabel="tokens", ylabel="cross entropy")
axes[2].grid(alpha=0.25)
axes[2].legend(fontsize=7)
fig.tight_layout()
fig.savefig(RUN_ROOT / "benchmark_summary.png", dpi=180)
plt.show()

## Export a compact results archive

The archive excludes token caches and resumable model checkpoints. It contains
the environment manifest, summaries, every per-step timing row, evaluations,
checkpoint timings, and the plot.

In [ ]:
EXPORT_PATH = PROJECT_ROOT / "logrms_50m_benchmark_results.zip"
with zipfile.ZipFile(EXPORT_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_ROOT.rglob("*"):
        if not path.is_file() or path.name in {"checkpoint.pt", "checkpoint.pt.tmp"}:
            continue
        archive.write(path, path.relative_to(RUN_ROOT.parent))

print({
    "results_csv": str(RUN_ROOT / "run_summary.csv"),
    "results_archive": str(EXPORT_PATH),
    "note": "Use total wall, full-step, and optimizer-step columns together; they answer different speed questions.",
})